# 07 — Analysis Sample Summary

Describes the datasets used in the DiD analysis.
Outputs: `../output/data_summary.tex`, `../output/summary_gallup.tex`

In [5]:
import pandas as pd
import gdown
import os

os.makedirs('../output', exist_ok=True)

# Load merged dataset (UN voting + aid + GDP from 01_load_clean.ipynb)
if not os.path.exists('df_clean.csv'):
    gdown.download(id='1Sw8kegqF-n9TIbTjc9Nq-n0wO4WtoGor', output='df_clean.csv', quiet=False)

df = pd.read_csv('df_clean.csv')

# Replicate treatment assignment from 02_analyze_un.ipynb
aid_reliance = (
    df[df['year'] == 2024]
    .groupby('ms_name')['aid_gdp_ratio']
    .mean()
    .fillna(0)
    .reset_index()
)
threshold = aid_reliance['aid_gdp_ratio'].quantile(0.75)
treated_countries = aid_reliance[aid_reliance['aid_gdp_ratio'] >= threshold]['ms_name']
df['treated'] = df['ms_name'].isin(treated_countries).astype(int)

# Aggregate to country-year level (unit of analysis in DiD)
did_df = df.groupby(['ms_name', 'year', 'treated']).agg(
    align_china=('align_china', 'mean'),
    align_us=('align_us', 'mean'),
).reset_index()

# Summary stats
n_obs        = len(did_df)
year_min     = int(did_df['year'].min())
year_max     = int(did_df['year'].max())
n_countries  = did_df['ms_name'].nunique()
n_treated    = did_df[did_df['treated'] == 1]['ms_name'].nunique()
n_control    = did_df[did_df['treated'] == 0]['ms_name'].nunique()

print(f"Observations (country-year):  {n_obs:,}")
print(f"Years:                        {year_min}\u2013{year_max}")
print(f"Countries:                    {n_countries}")
print(f"  Treated (top 25% aid/GDP):  {n_treated}")
print(f"  Control:                    {n_control}")
print(f"Treatment threshold:          {threshold:.3f}% of GDP")

Observations (country-year):  4,915
Years:                        2000–2025
Countries:                    192
  Treated (top 25% aid/GDP):  48
  Control:                    144
Treatment threshold:          0.533% of GDP


In [6]:
# Build and export summary table
summary = pd.DataFrame([{
    'Observations (country-year)': f'{n_obs:,}',
    'Years':                       f'{year_min}\u2013{year_max}',
    'Countries':                   n_countries,
    'Treated countries':           n_treated,
    'Control countries':           n_control,
}])

# Transpose for a cleaner single-column display
summary_t = summary.T.reset_index()
summary_t.columns = ['Statistic', 'Value']

print(summary_t.to_string(index=False))

latex = summary_t.to_latex(
    index=False,
    escape=False,
    column_format='lr',
    caption='UN Voting and Foreign Assistance: Analysis Sample',
    label='tab:data_summary',
)

with open('../output/data_summary.tex', 'w') as f:
    f.write(latex)

print("\nSaved \u2192 ../output/data_summary.tex")
summary_t

                  Statistic     Value
Observations (country-year)     4,915
                      Years 2000–2025
                  Countries       192
          Treated countries        48
          Control countries       144

Saved → ../output/data_summary.tex


,Statistic,Value
0,Observations (country-year),"4,915"
1,Years,2000–2025
2,Countries,192
3,Treated countries,48
4,Control countries,144


---
## 2. Gallup Approval Ratings — Analysis Sample

Summary statistics for the Gallup dataset used in the Difference-in-Differences approval analysis (`03_approval_ratings_load_clean.ipynb` + `04_approval_ratings_analysis.ipynb`).

The Gallup sample is smaller than the UN voting sample because:
- Gallup polling started in 2006 (vs. 2000 for UN votes)
- Non-UN territories (Kosovo, HK, Taiwan, etc.) are excluded
- The inner join on country-year drops Venezuela (no 2024 aid record) and Afghanistan

In [7]:
# ── Gallup approval ratings (final analysis sample from 03/04) ───────────────
if not os.path.exists('gallup_did_clean.csv'):
    gdown.download(id='17Pe4hqbilLjuSKkx6zEs_z-TQOgthRvF', output='gallup_did_clean.csv', quiet=False)

gal = pd.read_csv('gallup_did_clean.csv')

g_n_obs       = len(gal)
g_year_min    = int(gal['year'].min())
g_year_max    = int(gal['year'].max())
g_n_countries = gal['country'].nunique()
g_n_treated   = gal[gal['treated'] == 1]['country'].nunique()
g_n_control   = gal[gal['treated'] == 0]['country'].nunique()

print(f"Observations (country-year):  {g_n_obs:,}")
print(f"Years:                        {g_year_min}–{g_year_max}")
print(f"Countries:                    {g_n_countries}")
print(f"  Treated (top 25% aid/GDP):  {g_n_treated}")
print(f"  Control:                    {g_n_control}")

# Build transposed Statistic / Value table — same format as data_summary.tex
g_summary = pd.DataFrame([{
    'Observations (country-year)': f'{g_n_obs:,}',
    'Years':                       f'{g_year_min}–{g_year_max}',
    'Countries':                   g_n_countries,
    'Treated countries':           g_n_treated,
    'Control countries':           g_n_control,
}])

g_summary_t = g_summary.T.reset_index()
g_summary_t.columns = ['Statistic', 'Value']

print(g_summary_t.to_string(index=False))

latex = g_summary_t.to_latex(
    index=False,
    escape=False,
    column_format='lr',
    caption='Gallup Leadership Approval: Analysis Sample',
    label='tab:summary_gallup',
)

with open('../output/summary_gallup.tex', 'w') as f:
    f.write(latex)

print("\nSaved → ../output/summary_gallup.tex")
g_summary_t

Observations (country-year):  2,352
Years:                        2006–2025
Countries:                    154
  Treated (top 25% aid/GDP):  42
  Control:                    112
                  Statistic     Value
Observations (country-year)     2,352
                      Years 2006–2025
                  Countries       154
          Treated countries        42
          Control countries       112

Saved → ../output/summary_gallup.tex


,Statistic,Value
0,Observations (country-year),"2,352"
1,Years,2006–2025
2,Countries,154
3,Treated countries,42
4,Control countries,112
